In [1]:

import pandas as pd

files = {
    "DEGs": ("top500_DEGs.csv", ","),
    "LASSO": ("top20_biomarkers.csv", ","),
    "RF": ("rf_top20.csv", ","),
    "DGIdb": ("dgidb_gene_interaction_summary.csv", ","),
    "STRING": ("string_node_degrees_15_nodes (1).tsv", "\t"),
}

for name, (path, sep) in files.items():
    df = pd.read_csv(path, sep=sep)
    print("=====", name, df.shape)
    print(list(df.columns))
    print(df.head(3).to_string())
    print()


FileNotFoundError: [Errno 2] No such file or directory: 'top500_DEGs.csv'

In [2]:
import os

print(os.listdir("/content"))

['.config', 'top20_biomarkers (1).csv', 'rf_top20.csv', 'top500_DEGs (1).csv', 'dgidb_gene_interaction_summary.csv', 'string_node_degrees_15_nodes (1).tsv', 'sample_data']


In [3]:
import pandas as pd

files = {
    "DEGs": ("top500_DEGs (1).csv", ","),
    "LASSO": ("top20_biomarkers (1).csv", ","),
    "RF": ("rf_top20.csv", ","),
    "DGIdb": ("dgidb_gene_interaction_summary.csv", ","),
    "STRING": ("string_node_degrees_15_nodes (1).tsv", "\t"),
}

for name, (path, sep) in files.items():
    df = pd.read_csv(path, sep=sep)
    print("=====", name, df.shape)
    print(list(df.columns))
    print(df.head(3).to_string())
    print()

===== DEGs (500, 4)
['gene', 'logFC', 'p', 'FDR']
          gene     logFC             p           FDR
0         FZD9  1.551756  2.537657e-75  5.495296e-71
1  COL18A1-AS1 -1.014544  1.090431e-72  1.180664e-68
2         SKA1  1.165957  1.338017e-67  9.658251e-64

===== LASSO (13, 5)
['Gene', 'LASSO_Coef', 'Abs_Coef', 'TNBC_minus_NonTNBC', 'Direction']
          Gene  LASSO_Coef  Abs_Coef  TNBC_minus_NonTNBC         Direction
0  COL18A1-AS1   -1.568535  1.568535           -1.014544  Down in TNBC (-)
1     AX746627   -0.869316  0.869316           -0.862589  Down in TNBC (-)
2         FZD9    0.415668  0.415668            1.551756    Up in TNBC (+)

===== RF (20, 2)
['Gene', 'RF_Importance']
           Gene  RF_Importance
0      AX746627       0.071866
1  RP11-55K13.1       0.047406
2          SKA1       0.034286

===== DGIdb (15, 2)
['Gene', 'Interaction_Count']
    Gene  Interaction_Count
0   FZD9                  0
1   SKA1                  0
2  KCNQ4                 21

===== STRING (1

In [4]:
import pandas as pd

# Load files
deg = pd.read_csv("top500_DEGs (1).csv")
lasso = pd.read_csv("top20_biomarkers (1).csv")
rf = pd.read_csv("rf_top20.csv")
dgidb = pd.read_csv("dgidb_gene_interaction_summary.csv")
string = pd.read_csv("string_node_degrees_15_nodes (1).tsv", sep="\t")

# Standardize gene column names
deg = deg.rename(columns={"gene": "Gene"})
string = string.rename(columns={"#node": "Gene"})

# Clean gene names
for df in [deg, lasso, rf, dgidb, string]:
    df["Gene"] = df["Gene"].astype(str).str.strip()

# Start with all genes selected by LASSO or Random Forest
candidates = pd.DataFrame({
    "Gene": pd.unique(pd.concat([lasso["Gene"], rf["Gene"]]))
})

# Merge evidence from each analysis
final = candidates.merge(deg, on="Gene", how="left")
final = final.merge(lasso, on="Gene", how="left")
final = final.merge(rf, on="Gene", how="left")
final = final.merge(dgidb, on="Gene", how="left")
final = final.merge(
    string[["Gene", "node_degree"]].drop_duplicates("Gene"),
    on="Gene",
    how="left"
)

# Add selection indicators
final["LASSO_Selected"] = final["LASSO_Coef"].notna()
final["RF_Selected"] = final["RF_Importance"].notna()

# Keep missing evidence distinguishable from zero
final = final.sort_values(
    by=["LASSO_Selected", "RF_Selected", "FDR"],
    ascending=[False, False, True],
    na_position="last"
)

# Save
final.to_csv("final_candidate_table.csv", index=False)

print("Final table shape:", final.shape)
print(final.to_string(index=False))
print("\nSaved: final_candidate_table.csv")

Final table shape: (27, 13)
        Gene     logFC            p          FDR  LASSO_Coef  Abs_Coef  TNBC_minus_NonTNBC        Direction  RF_Importance  Interaction_Count  node_degree  LASSO_Selected  RF_Selected
        FZD9  1.551756 2.537657e-75 5.495296e-71    0.415668  0.415668            1.551756   Up in TNBC (+)       0.032254                0.0          0.0            True         True
 COL18A1-AS1 -1.014544 1.090431e-72 1.180664e-68   -1.568535  1.568535           -1.014544 Down in TNBC (-)       0.033381                NaN          NaN            True         True
    AX746627 -0.862589 4.163379e-59 2.253949e-55   -0.869316  0.869316           -0.862589 Down in TNBC (-)       0.071866                NaN          NaN            True         True
RP11-55K13.1 -0.495239 1.261101e-54 3.034348e-51   -0.176110  0.176110           -0.495239 Down in TNBC (-)       0.047406                NaN          NaN            True         True
   LOC338667 -1.053466 4.901792e-40 5.897128e-37   -

In [5]:
from google.colab import files
files.download("final_candidate_table.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [6]:
import pandas as pd

known = pd.DataFrame({
    "gene":   ["ESR1", "PGR", "GATA3", "FOXA1", "ERBB2", "KRT5", "BCL11A"],
    "logFC":  [-0.524282, -2.238412, -2.559842, -2.004057, -0.581972, 1.850510, 1.541273],
    "FDR":    [1.688440e-17, 3.185310e-15, 5.569195e-24, 5.523808e-20,
               1.305983e-06, 9.072937e-12, 4.334556e-19],
})
known["direction_in_TNBC"] = known["logFC"].apply(lambda v: "Up" if v > 0 else "Down")

known.to_csv("known_marker_results.csv", index=False)
print(known.to_string())

     gene     logFC           FDR direction_in_TNBC
0    ESR1 -0.524282  1.688440e-17              Down
1     PGR -2.238412  3.185310e-15              Down
2   GATA3 -2.559842  5.569195e-24              Down
3   FOXA1 -2.004057  5.523808e-20              Down
4   ERBB2 -0.581972  1.305983e-06              Down
5    KRT5  1.850510  9.072937e-12                Up
6  BCL11A  1.541273  4.334556e-19                Up


In [7]:
from google.colab import files
files.download("known_marker_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>